# Modelos de Aprendizado de Máquina
## Análise de Dados de Compliance Pública - Tese de MBA

**Objetivo:** Construir modelos preditivos para análise de compliance:
- Modelos de regressão (prever taxa de sanções)
- Modelos de classificação (risco de compliance alto/baixo)
- Análise de importância de features
- Avaliação e comparação de modelos

In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


## Passo a passo (estilo aula)

1. Pacotes e configuração do ambiente
2. Reprodutibilidade
3. Carregamento dos dados
4. Blocos de análise
5. Resumo e interpretação


# Pacotes


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Try local loader first, fallback to S3 if available
from src.analysis.local_data_loader import LocalGoldDataLoader as GoldDataLoader

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)
pd.set_option('display.float_format', '{:.4f}'.format)


In [ ]:
import matplotlib as mpl
mpl.rcParams['axes.formatter.useoffset'] = False
mpl.rcParams['axes.formatter.limits'] = (-99, 99)


# Reprodutibilidade


In [ ]:
import os
import random

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Semente de reprodutibilidade fixada em {SEED}")


In [ ]:
import json as _json

_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}

S3_BUCKET_NAME = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))
AWS_PROFILE = os.environ.get('AWS_PROFILE', _rtcfg.get('aws', {}).get('profile', None))


## 1. Carregar e Preparar Dados

In [ ]:
from src.analysis.pt_br_loader import GoldDataLoaderPtBr as GoldDataLoader
loader = GoldDataLoader()

# --- Análise em nível municipal (N ~= 5.570) -----------------------------
# Agora carregamos o dataset municipal `analise_compliance_municipio` em vez
# do dataset estadual `analise_compliance` (N=27). A granularidade municipal
# garante poder estatístico adequado para correlação, OLS e ML abaixo.
#
# Para manter compatibilidade com o restante do notebook:
#   1. Renomeamos colunas do nível municipal para os antigos nomes estaduais
#      (ex.: `populacao_2022` -> `populacao`), preservando células a jusante.
#   2. Recriamos as dummies regionais humanamente legíveis (is_norte, is_nordeste,
#      is_sudeste, is_sul, is_centro_oeste) com os mesmos nomes do dataset
#      estadual.
#   3. Adicionamos dummies de estado (is_state_<código-IBGE>) como features extras.
df = loader.load_dataset('analise_compliance_municipio')
df = df.rename(columns={
    'populacao_2022': 'populacao',
    'taxa_alfabetizacao_2022': 'taxa_alfabetizacao_media',
    'renda_media_2022': 'renda_media',
})

REGION_NAME_TO_DUMMY = {
    'Norte': 'is_norte',
    'Nordeste': 'is_nordeste',
    'Sudeste': 'is_sudeste',
    'Sul': 'is_sul',
    'Centro-Oeste': 'is_centro_oeste',
}
for _rname, _col in REGION_NAME_TO_DUMMY.items():
    df[_col] = (df['nome_regiao'] == _rname).astype('Int64')
REGION_DUMMY_COLS = list(REGION_NAME_TO_DUMMY.values())

state_dummies = pd.get_dummies(df['codigo_estado'], prefix='is_state').astype('Int64')
df = pd.concat([df, state_dummies], axis=1)
STATE_DUMMY_COLS = list(state_dummies.columns)

print(f"Carregadas {len(df):,} observações (municípios em {df['codigo_estado'].nunique()} estados)")
print(f"Dummies regionais: {REGION_DUMMY_COLS}")
print(f"Dummies de estado: {len(STATE_DUMMY_COLS)} colunas (primeira: {STATE_DUMMY_COLS[0]}, última: {STATE_DUMMY_COLS[-1]})")
df.head()

In [ ]:
# Colunas de features em nível MUNICIPAL.
# - removido `num_municipios` (constante = 1 nessa granularidade)
# - removido `is_sudeste` como região-base implícita (dummy trap)
# - adicionado `log_total_transferencias` se presente -- central para a pergunta do TCC
feature_cols = ['log_renda', 'taxa_alfabetizacao_media', 'log_populacao',
                'is_norte', 'is_nordeste', 'is_sul', 'is_centro_oeste']
if 'log_total_transferencias' in df.columns:
    feature_cols.append('log_total_transferencias')

# Remove linhas com NaN em features ou alvo (nível municipal tem colunas nullable).
_mask = df[feature_cols + ['sancoes_por_100k']].notna().all(axis=1)
X = df.loc[_mask, feature_cols].copy().astype(float)
y_regression = df.loc[_mask, 'sancoes_por_100k'].astype(float)

print(f"Formato de features: {X.shape}  (N={len(X):,} municípios após remover NaN)")
print(f"Formato do alvo: {y_regression.shape}")
print(f"\nFeatures: {list(X.columns)}")

## 2. Modelos de Regressão - Prever Taxa de Sanções

### 2.1 Divisão Treino-Teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_regression, test_size=0.25, random_state=42
)

print(f"Conjunto de treino: {len(X_train)} amostras")
print(f"Conjunto de teste: {len(X_test)} amostras")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)


### 2.2 Treinamento e Comparação de Modelos

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
}

results = []

for name, model in models.items():
    print(f"Treinando {name}...")
    
    if name in ['Ridge', 'Lasso', 'ElasticNet']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    })

results_df = pd.DataFrame(results).sort_values('R²', ascending=False)

print("\n" + "=" * 80)
print("COMPARAÇÃO DE MODELOS DE REGRESSÃO")
print("=" * 80)
display(results_df)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].barh(results_df['Model'], results_df['R²'], color='steelblue', alpha=0.7)
axes[0].set_xlabel('R² Score')
axes[0].set_title('Desempenho do Modelo: R²', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

axes[1].barh(results_df['Model'], results_df['RMSE'], color='coral', alpha=0.7)
axes[1].set_xlabel('RMSE')
axes[1].set_title('Desempenho do Modelo: RMSE', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

axes[2].barh(results_df['Model'], results_df['MAE'], color='lightgreen', alpha=0.7)
axes[2].set_xlabel('MAE')
axes[2].set_title('Desempenho do Modelo: MAE', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


### 2.3 Melhor Modelo - Análise Detalhada

In [ ]:
nome_melhor_modelo = results_df.iloc[0]['Model']
melhor_modelo = models[nome_melhor_modelo]

print(f"Melhor Modelo: {nome_melhor_modelo}")
print("=" * 70)

if nome_melhor_modelo in ['Ridge', 'Lasso', 'ElasticNet']:
    y_pred_best = melhor_modelo.predict(X_test_scaled)
else:
    y_pred_best = melhor_modelo.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y_test, y_pred_best, alpha=0.6, s=100)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Sanções por 100 mil Reais', fontsize=12)
axes[0].set_ylabel('Sanções por 100 mil Previstas', fontsize=12)
axes[0].set_title(f'{nome_melhor_modelo}: Real vs Previsto', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

residuals = y_test - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.6, s=100)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Sanções por 100 mil Previstas', fontsize=12)
axes[1].set_ylabel('Resíduos', fontsize=12)
axes[1].set_title(f'{nome_melhor_modelo}: Gráfico de Resíduos', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 2.4 Importância de Features (Modelos Baseados em Árvore)

In [ ]:
rf_model = models['Random Forest']

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importância': rf_model.feature_importances_
}).sort_values('Importância', ascending=False)

print("Importância das Features (Random Forest)")
print("=" * 70)
display(feature_importance)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importância'], color='teal', alpha=0.7)
plt.xlabel('Importância', fontsize=12)
plt.title('Importância das Features - Random Forest', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


### 2.5 Validação Cruzada

In [ ]:
cv_results = []
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    if name in ['Ridge', 'Lasso', 'ElasticNet']:
        X_cv = scaler.fit_transform(X)
    else:
        X_cv = X
    
    scores = cross_val_score(model, X_cv, y_regression, cv=kfold, 
                            scoring='r2', n_jobs=-1)
    
    cv_results.append({
        'Model': name,
        'Média R²': scores.mean(),
        'Std R²': scores.std(),
        'Min R²': scores.min(),
        'Max R²': scores.max()
    })

cv_df = pd.DataFrame(cv_results).sort_values('Média R²', ascending=False)

print("5-Fold Resultados da Validação Cruzada")
print("=" * 80)
display(cv_df)


## 3. Modelos de Classificação - Risco de Compliance Alto/Baixo

### 3.1 Criar Variável Alvo Binária

In [ ]:
# Definição do alvo binário (nível municipal).
#
# RESSALVA SOBRE OS DADOS: no snapshot Gold atual ~78% dos municípios têm
# `num_sancoes == 0`, então a mediana de `sancoes_por_100k` é 0. Usar
# `> mediana` como fronteira colapsa para "teve ao menos uma sanção", que
# É um alvo binário significativo (o município apareceu alguma vez em
# CEIS / CNEP / CEPIM?). Definimos então:
#     high_risk == 1  se  o município registra ao menos uma sanção.
# O balanceamento fica ~22/78, por isso os classificadores abaixo usam
# class_weight='balanced' (e acompanhamos ROC-AUC / F1, não só acurácia).
df_sub = df.loc[_mask].copy()
df_sub['high_risk'] = (df_sub['num_sancoes'] > 0).astype(int)

print("Alvo binário: tem ao menos uma sanção registrada")
print(f"Distribuição de classes:")
print(df_sub['high_risk'].value_counts())
print(f"\nBalanceamento de classes:")
print(df_sub['high_risk'].value_counts(normalize=True).round(3))

y_classification = df_sub['high_risk'].copy()

### 3.2 Treinar Modelos de Classificação

In [ ]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X, y_classification, test_size=0.25, random_state=42, stratify=y_classification
)

scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

print(f"Conjunto de treino: {len(X_train_clf)} amostras")
print(f"Conjunto de teste: {len(X_test_clf)} amostras")


In [ ]:
# class_weight='balanced' compensa o desbalanceamento ~78/22 (apenas 22%
# dos municípios têm sanção registrada). Sem isso, os classificadores preveriam
# trivialmente 0 em tudo e teriam 78% de acurácia errando todos os positivos.
clf_models = {
    'Regressão Logística': LogisticRegression(random_state=42, max_iter=1000,
                                              class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5,
                                            random_state=42, n_jobs=-1,
                                            class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100,
                                                    max_depth=3, random_state=42),
}

clf_results = []

for name, model in clf_models.items():
    print(f"\nTreinando {name}...")

    if name == 'Regressão Logística':
        model.fit(X_train_clf_scaled, y_train_clf)
        y_pred = model.predict(X_test_clf_scaled)
        y_pred_proba = model.predict_proba(X_test_clf_scaled)[:, 1]
    else:
        model.fit(X_train_clf, y_train_clf)
        y_pred = model.predict(X_test_clf)
        y_pred_proba = model.predict_proba(X_test_clf)[:, 1]

    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

    accuracy = accuracy_score(y_test_clf, y_pred)
    precision = precision_score(y_test_clf, y_pred, zero_division=0)
    recall = recall_score(y_test_clf, y_pred, zero_division=0)
    f1 = f1_score(y_test_clf, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test_clf, y_pred_proba)

    clf_results.append({
        'Modelo': name,
        'Acurácia': accuracy,
        'Precisão': precision,
        'Recall': recall,
        'F1': f1,
        'ROC-AUC': roc_auc,
    })

    print(f"\n{name} - Relatório de Classificação:")
    print(classification_report(y_test_clf, y_pred,
                                target_names=['Sem sanções', 'Com sanções'],
                                zero_division=0))

clf_results_df = pd.DataFrame(clf_results).sort_values('ROC-AUC', ascending=False)

print("\n" + "=" * 80)
print("COMPARAÇÃO DE MODELOS DE CLASSIFICAÇÃO (desbalanceado, class_weight='balanced')")
print("=" * 80)
display(clf_results_df)

### 3.3 Matrizes de Confusão

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, model) in enumerate(clf_models.items()):
    if name == 'Logistic Regression':
        y_pred = model.predict(X_test_clf_scaled)
    else:
        y_pred = model.predict(X_test_clf)
    
    cm = confusion_matrix(y_test_clf, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Baixo Risco', 'Alto Risco'],
                yticklabels=['Baixo Risco', 'Alto Risco'])
    axes[idx].set_title(f'{name}\nMatriz de Confusão', fontweight='bold')
    axes[idx].set_ylabel('Real')
    axes[idx].set_xlabel('Previsto')

plt.tight_layout()
plt.show()


### 3.4 Curvas ROC

In [ ]:
plt.figure(figsize=(10, 8))

for name, model in clf_models.items():
    if name == 'Logistic Regression':
        y_pred_proba = model.predict_proba(X_test_clf_scaled)[:, 1]
    else:
        y_pred_proba = model.predict_proba(X_test_clf)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_test_clf, y_pred_proba)
    auc = roc_auc_score(y_test_clf, y_pred_proba)
    
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=2)
plt.xlabel('Taxa de Falso Positivo', fontsize=12)
plt.ylabel('Taxa de Verdadeiro Positivo', fontsize=12)
plt.title('Curva ROCs - Compliance Risk Classification', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Interpretação do Modelo

### 4.1 Coeficientes da Regressão Logística

In [ ]:
lr_model = clf_models['Logistic Regression']

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0],
    'Odds Ratio': np.exp(lr_model.coef_[0])
}).sort_values('Coefficient', key=abs, ascending=False)

print("Logistic Regression Coefficients")
print("=" * 70)
display(coef_df)

plt.figure(figsize=(10, 6))
colors = ['red' if c < 0 else 'green' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, alpha=0.7)
plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Valor do Coeficiente', fontsize=12)
plt.title('Logistic Regression Coefficients\n(Red = Negative, Green = Positive)', 
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


### 4.2 Importância de Features (Classificação)

In [ ]:
rf_clf = clf_models['Random Forest']

feature_importance_clf = pd.DataFrame({
    'Feature': X.columns,
    'Importância': rf_clf.feature_importances_
}).sort_values('Importância', ascending=False)

print("Importância das Features (Random Forest Classifier)")
print("=" * 70)
display(feature_importance_clf)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_clf['Feature'], feature_importance_clf['Importância'], 
         color='purple', alpha=0.7)
plt.xlabel('Importância', fontsize=12)
plt.title('Importância das Features - Random Forest Classifier', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


## 5. Resumo e Recomendações

### 5.1 Modelos de Regressão
- **Melhor modelo**: ElasticNet, selecionado via validação cruzada entre Regressão Linear, Ridge, Lasso, ElasticNet, Árvore de Decisão, Random Forest e Gradient Boosting.
- **Principais preditores**: Log da renda e dummies regionais (Norte, Nordeste) são as features mais importantes para prever sanções por 100 mil.
- **Importância de features do Random Forest** confirma a renda como preditor dominante, seguida por indicadores de população e alfabetização.
- **Limitação**: Com apenas 27 observações (estados), todos os modelos enfrentam alta variância e generalização limitada.

### 5.2 Modelos de Classificação
- **Alvo binário**: Estados classificados como alto risco (acima da mediana de sanções por 100 mil de 12,22) vs baixo risco.
- **Balanceamento de classes**: Quase balanceado (14 baixo risco, 13 alto risco).
- **Regressão Logística** obteve o melhor desempenho F1 (F1 ponderado = 0,51), mas com recall fraco na classe Baixo Risco (0,25).
- **Random Forest e Gradient Boosting** tiveram desempenho inferior (acurácia 0,43 e 0,29 respectivamente), provavelmente por overfitting no dataset pequeno.
- **Resumo**: O desempenho de classificação é fraco em todos os modelos devido ao tamanho amostral pequeno (n = 27). Resultados devem ser tratados como exploratórios.

### 5.3 Insights Principais
- Renda é consistentemente o preditor mais forte de taxas de sanções em todas as abordagens de modelagem.
- Efeitos regionais (Norte, Nordeste) persistem após controlar por indicadores socioeconômicos, sugerindo fatores institucionais ou de governança em jogo.
- O Distrito Federal permanece como outlier forte que influencia todos os modelos.

### 5.4 Limitações e Ressalvas
- **Amostra pequena** (n = 27): Insuficiente para treinamento robusto de modelos ML. Validação cruzada ajuda, mas não compensa totalmente.
- **Correlação, não causalidade**: Renda mais alta pode refletir maior capacidade institucional de detectar irregularidades, não mais irregularidades de fato.
- **Agregação em nível estadual** mascara variação intra-estadual — análise em nível municipal (NB04) fornece maior granularidade.
- **Sem dimensão temporal**: Modelos são recortes transversais, não preditivos de tendências futuras de sanções.
